# Implement The Decoder Block

![Main Encoder Block](../showcase_images/from_paper/Decoder.png)

- **Decoder Stack**:
  - Composed of a stack of $Nx=6$ identical decoding layers.
  - The Decoder first sees its own output shifted right, then sees the output from the Encoder.
  - Has both the layers the Encoder has, but also has a **Masked Multi-Head Attention** that prevents cheating by helping the Decoder understand the sequence it has generated so far, allowing it to only see pass tokens in the output sentence.

In [8]:
import torch.nn as nn
import torch
import copy

from residual_con_layer_norm import ResidualConnection, LayerNorm
from multi_head_attention import Multi_Head_Attention
from FeedForwardNetwork import FeedForwardNetwork

In [ ]:
def make_decoder_mask(tgt_tokens, pad_token):
    """
    Hides padding and future tokens (no-peeking), so the model doesn't cheat. This is specifically for the Masked Multi-Head Attention in the DecoderLayer.

    Args:
        tgt_tokens: The target tokens from the output of the decoder.
    """
    # Hide the padding tokens, e.g., "The brown rabbit ate the apple. <Padding> <Padding> <Padding>", hides the <Padding> tokens.
    padding_mask = (tgt_tokens != pad_token).unsqueeze(1).unsqueeze(2)

    # Hide future input tokens, so the model doesn't cheat.
    seq_len = tgt_tokens.size(1)
    no_peak_mask = torch.tril(
        torch.ones(seq_len, seq_len, device=tgt_tokens.device)
    ).bool()

    # Combine
    return padding_mask & no_peak_mask

In [28]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff, dropout):
        """
        A single layer of the Decoder Stack.
        """
        super().__init__()
        self.d_model = d_model

        # Masked Attention
        self.masked_mha = Multi_Head_Attention(d_model, h, dropout)
        # Normal Attention
        self.mha = Multi_Head_Attention(d_model, h, dropout)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)

        # Three residual connections: One for the Masked Multi-Head Attention, the other for the normal Multi-Head Attention, and the last for the Feed Forward
        self.residual_1_masked_mha = ResidualConnection(d_model, dropout)
        self.residual_2_mha = ResidualConnection(d_model, dropout)
        self.residual_3_ffn = ResidualConnection(d_model, dropout)

    def forward(self, x, encoder_output, padding_mask, no_peak_mask):
        """
        Args:
            x: Target sequence (from Decoder)
            encoder_output: The Encoder's final output
            padding_mask: The padding for sequences.
            no_peak_mask: The mask for the Masked Multi-Head Attention.
        """

        # Masked Attention Sublayer
        x = self.residual_1_masked_mha(
            x, lambda x: self.masked_mha(x, x, x, no_peak_mask)
        )

        # Normal Attention sublayer connection between Encoder-Decoder
        x = self.residual_2_mha(
            x, lambda x: self.mha(x, encoder_output, encoder_output, padding_mask)
        )

        # Feed-Forward Sublayer
        x = self.residual_3_ffn(x, self.ffn)
        return x

In [29]:
from model_utils import clones


class Decoder(nn.Module):
    def __init__(self, layer: DecoderLayer, N):
        """
        The full Decoder that is a Nx stack of DecoderLayer()

        Args:
            layer: A single DecoderLayer().
            N: The stack size of DecoderLayer()
        """
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.d_model)

    def forward(self, x, encoder_output, padding_mask, no_peak_mask):
        """
        Arg:
            encoder_output: The Encoder's final output
            padding_mask: The padding for sequences. Example with 3 paddings sequence: "The brown rabbit ate the apple. <Padding> <Padding> <Padding>". This mask hides the paddings from the Encoder, so it only sees non-padding tokens.
            no_peak_mask: The mask for the Masked Multi-Head Attention, this is different from padding_mask. Without this mask, the model would simply cheat by looking at the next work in the input and copying it. Note: This is for training, it is different during inference. Note: The no_peak_mask also includes the padding_mask.
        """
        for layer in self.layers:
            x = layer(x, encoder_output, padding_mask, no_peak_mask)
        return self.norm(x)

In [30]:
def test():
    print("\n\nTesting the Decoder...")
    d_model, h, d_ff, N, dropout = 512, 8, 2048, 6, 0.1
    batch_size, seq_len = 2, 10

    base_dec_layer = DecoderLayer(d_model, h, d_ff, dropout)
    decoder = Decoder(base_dec_layer, N)
    
    # Inputs
    dummy_input = torch.randn(batch_size, seq_len, d_model)
    encoder_output = torch.randn(batch_size, seq_len, d_model)

    # Token IDs for mask generation
    target_tokens = torch.ones(batch_size, seq_len)
    target_tokens[:, -3:] = 0 # last 3 are padding ie: "<padding>"

    # Mask: 
    # no_peak_mask= padding_mask + no_peak_mask
    no_peak_mask = make_decoder_mask(target_tokens, pad_token=0)
    # Create a simple padding Mask, the last 3 tokens are padding
    padding_mask = (target_tokens != 0).unsqueeze(1).unsqueeze(2)

    output = decoder(dummy_input, encoder_output, padding_mask, no_peak_mask)
    print(f"Output shape: {output.shape}")

    # assert output.shape == dummy_input.shape, "Output shape mismatch!"

    # Verify Gradient flow
    output.mean().backward()
    # Pick a parameter at the very beginning of the network (e.g., the first masked attention sublayer weights)
    start_param = decoder.layers[0].masked_mha.w_q.weight
    assert start_param.grad is not None, "Gradient did not reach the first layer!"
    assert torch.any(start_param.grad != 0), "Gradients are zero (Vanishing Gradient)!"
    decoder.zero_grad()
    print("Test Passed!")


test()



Testing the Decoder...
Output shape: torch.Size([2, 10, 512])
Test Passed!
